<a href="https://colab.research.google.com/github/ajtamayoh/In2Lab-ITM-at-IberLEF-NivELE-2026/blob/main/In2Lab_ITM_run_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In2Lab-ITM at IberLEF NivELE 2026: Automatic CEFR Level Classification for Spanish Learner Texts: A Comparative Study of a Pretrained LLM and Prompt Engineering

Antonio Jesus Tamayo Herrera (University of Antioquia, UdeA)

Juan Felipe Zuluaga Molina (Institución Universitaria ITM)

Correspondence: antonio.tamayo@udea.edu.co

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/data/test.csv", index_col="id")
print(df.head())

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# --- Configuración ---
model_name = "pymlex/roberta-spanish-cefr"
labels = ["A1", "A2", "B1", "B2", "C1"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Cargar modelo y tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

# --- Función de predicción ---
def predict_batch(texts, batch_size=16, max_length=256):
    preds = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            batch_preds = torch.argmax(logits, dim=1).cpu().numpy()

        preds.extend(batch_preds)

    return preds

# --- Asumiendo que ya tienes df con columnas ['id', 'text'] ---
# Ejemplo:
# df = pd.read_csv("test.csv")

# --- Ejecutar predicción ---
texts = df["text"].tolist()
pred_indices = predict_batch(texts)

# Mapear índices a labels
df["label"] = [labels[i] for i in pred_indices]

# --- Resultado ---
print(df.head())


In [ ]:
submission = df[["label"]]
submission.to_csv("/content/drive/MyDrive/results/run_1/submission.csv", index="id")